# PIC benchmark of particle-core model

In [ ]:
import argparse
import os
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import scipy.constants
import scipy.signal
from ipywidgets import interact
from tqdm.notebook import trange

# PyORBIT
from orbit.core.bunch import Bunch
from orbit.core.spacecharge import SpaceChargeCalc2p5D
from orbit.space_charge.sc2p5d import setSC2p5DAccNodes
from orbit.teapot import ContinuousLinearFocusingTEAPOT
from orbit.teapot import TEAPOT_Lattice
from orbit.utils.consts import mass_proton

# local
from tools import pcm
from tools.diag import BunchHistogramCalculator
from tools.utils import intensity_from_perveance
from tools.utils import samp_dist

In [ ]:
plt.style.use("style.mplstyle")

Set simulation parameters.

In [ ]:
sigma0 = np.radians(80.0)  # phase advance [deg]
eta = 0.5  # tune depression ratio (k / k0)
radius_frac = 0.6095  # initial beam radius / matched radius 
emittance = 1e-6  # rms emittance (times 4) [m rad]
bunch_length = 10.0  # bunch length [m]
kin_energy = 0.0025  # kinetic energy [GeV]

# Calculate depressed phase advance
sigma = sigma0 * eta
k0 = sigma0  # wavenumber (assume length = 1 [m])
k = k0 * eta  # depressed wavenumber

# Stationary KV equilibrium parameters
eq_radius = pcm.get_eq_radius(emittance, k)
perveance = pcm.get_eq_perveance(emittance, k, k0)

# Mismatched core radius
radius = eq_radius * radius_frac

# Construct 4 x 4 covariance matrix
cov_matrix = np.zeros((6, 6))
cov_matrix[0:4, 0:4] = pcm.get_eq_cov_matrix(radius, emittance)
cov_matrix[4, 4] = (bunch_length ** 2) / 12.0  # for uniform density
cov_matrix_init = cov_matrix.copy()

# Get beam intensity from perveance, kinetic energy, mass, and length.
intensity = intensity_from_perveance(perveance, kin_energy, mass_proton, bunch_length)

Find core oscillation wavelength from envelope tracking (ODE solver). For large mismatch this will deviate slightly from the breathing-mode frequency 
. Note that pcm.py uses dimensionless coordinates.

In [ ]:
# From small-amplitude perturbation
env_wave_pred = (2.0 * np.pi) / np.sqrt(2.0 * (1.0 + eta**2))

# Track envelope for ~20 core oscillations.
periods = 20
history = pcm.track(
    envelope=np.array([radius_frac, 0.0]),
    particles=np.array([[2.8, 0.0]]),
    eta=eta,
    t_max=(periods * env_wave_pred),
    t_steps=(periods * 100),
)

# Find the peaks
idx, _ = scipy.signal.find_peaks(history["r"])
env_wave_avg = np.mean(np.diff(history["t"][idx]))
env_wave_std = np.std(np.diff(history["t"][idx]))
print("wavelength * k0 (pred) = {:0.4f}".format(env_wave_pred))
print("wavelength * k0 (calc) = {:0.4f} +- {:0.4f}".format(env_wave_avg, env_wave_std))

# Correct wavelength
env_wave = env_wave_avg / k0

Place a test particle just outside the separatrix and track it for many core oscillations. This forms an approximate upper bound on the halo which can be compared to PIC simulations.

In [ ]:
history_pcm = pcm.track_strobe(
    envelope=np.array([radius_frac, 0.0]),
    particles=np.array([[2.8, 0.0]]),  # ~separatrix
    eta=eta,
    periods=500,
)

Set up the PyORBIT simulation.

In [ ]:
# Sample particles from 4D KV distribution.
particles = np.zeros((100_000, 6))
particles[:, :4] = samp_dist(
    size=particles.shape[0],
    name="kv", 
    cov_matrix=cov_matrix_init[:4, :4], 
    seed=14
)
particles[:, 4] = bunch_length * np.random.uniform(-0.5, 0.5, size=particles.shape[0])

# Create bunch object.
bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(kin_energy)
bunch.macroSize(intensity / particles.shape[0])
for i in range(particles.shape[0]):
    bunch.addParticle(*particles[i])

# Create an accelerator lattice with a continuous-focusing node.
lattice = TEAPOT_Lattice()
lattice.addNode(
    ContinuousLinearFocusingTEAPOT(length=env_wave, kq=k0**2, nparts=30)
)

# Add space charge nodes to the lattice.
sc_calc = SpaceChargeCalc2p5D(128, 128, 1)
sc_path_length_min = 0.001
sc_nodes = setSC2p5DAccNodes(lattice, sc_path_length_min, sc_calc)

In [ ]:
# Set histogram grid limits.
scale = np.array([eq_radius, eq_radius * k0])
xmax = 3.5 * scale
limits = list(zip(-xmax, xmax))

# Create histogram calculator. This grids the particles using PyORBIT `Grid` object.
hist_calc = BunchHistogramCalculator(
    axis=(0, 1),
    shape=(150, 150),
    limits=limits,
)

# Track the bunch.
histograms = []
for period in trange(25 + 1):
    if period > 0:
        lattice.trackBunch(bunch)

    histograms.append(hist_calc(bunch))

Plot the $x$-$x'$ projections in log scale down to $10^{-3}$ as a fraction of the peak density. The PC model result for a particle just outside the separatrix is shown in red.

In [ ]:
@interact(period=(0, len(histograms) - 1))
def update(period: int = 0) -> None:
    histogram = histograms[period].copy()
    values = histogram.values
    values = values / np.max(values)
    values = np.ma.log10(values)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.set_xlabel("$x / r_0$")
    ax.set_ylabel(r"$x' / k_0 r_0$")
    ax.scatter(
        history_pcm["particles"][..., 0],
        history_pcm["particles"][..., 1],
        ec="none",
        s=1,
        c="red",
        zorder=9999,
    )
    mesh = ax.pcolormesh(
        histogram.edges[0] / scale[0],
        histogram.edges[1] / scale[1],
        values.T,
        cmap="Greys",
        vmin=-3.0,
    )
    fig.colorbar(mesh)